# 🧠 How does a model see the world?

<a href="https://colab.research.google.com/github/racousin/scai_team/blob/main/how_models_see_the_world.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Every language model turns a text into a list of numbers — an **embedding**. That vector is, quite literally, how the model *sees* the text: two texts the model considers similar end up with nearby vectors.

In this notebook you will:

1. Take short bios — **your data team**, cartoon characters, politicians.
2. Ask **two different small models** from 🤗 Hugging Face to read them.
3. Project each model's vectors down to 2D and **look at the world through each model's eyes**.

▶️ **Runtime ▸ Run all**, then scroll down to the plots (≈1 min on the free CPU runtime). After that, come back and start editing the texts.

In [ ]:
!pip install -q sentence-transformers
!wget -q -O embedding_utils.py https://raw.githubusercontent.com/racousin/scai_team/main/embedding_utils.py
from embedding_utils import *

## 1. The texts

This is the only "data": one short bio per character, tagged with a `group`. Nothing else.

In [ ]:
ITEMS = [
    # --- the data team ------------------------------------------------------
    {"name": "Sam (data scientist)", "group": "data team",
     "text": "Data scientist who builds churn prediction models in Python, argues about feature engineering, and drinks too much coffee."},
    {"name": "Nadia (data engineer)", "group": "data team",
     "text": "Data engineer who keeps the pipelines running, moves terabytes between databases at night, and dreams in SQL."},
    {"name": "Leo (analyst)", "group": "data team",
     "text": "Business analyst who turns messy spreadsheets into dashboards and tells the story behind the numbers to management."},
    # 👇 ADD YOURSELF — uncomment, put your real name and a 1–2 sentence bio, re-run everything!
    # {"name": "YOUR NAME", "group": "data team",
    #  "text": "I am ..."},

    # --- cartoons -----------------------------------------------------------
    {"name": "Homer Simpson", "group": "cartoon",
     "text": "Lazy but lovable father from Springfield who works at a nuclear power plant, loves donuts and beer, and always says d'oh."},
    {"name": "Mickey Mouse", "group": "cartoon",
     "text": "Cheerful cartoon mouse with big round ears and red shorts who whistles, laughs, and stars in Disney adventures."},
    {"name": "Naruto", "group": "cartoon",
     "text": "Hyperactive young ninja from the Hidden Leaf Village who eats ramen, never gives up, and dreams of becoming Hokage."},
    {"name": "Elsa", "group": "cartoon",
     "text": "Ice queen from Arendelle with magical powers to freeze everything around her, who sings about letting it go."},
    {"name": "SpongeBob", "group": "cartoon",
     "text": "Optimistic yellow sea sponge who lives in a pineapple under the sea and flips burgers at the Krusty Krab."},

    # --- politicians ---------------------------------------------------------
    {"name": "Macron (EN)", "group": "politician",
     "text": "President of France, former investment banker, who leads the country from the Élysée Palace in Paris."},
    {"name": "Macron (FR)", "group": "politician",
     "text": "Président de la République française, ancien banquier d'affaires, qui dirige le pays depuis le palais de l'Élysée à Paris."},
    {"name": "Barack Obama", "group": "politician",
     "text": "Former President of the United States, known for his eloquent speeches, healthcare reform, and a Nobel Peace Prize."},
    {"name": "Angela Merkel", "group": "politician",
     "text": "Former Chancellor of Germany, a trained physicist who led Europe's largest economy for sixteen years."},
    {"name": "Nelson Mandela", "group": "politician",
     "text": "South African anti-apartheid revolutionary who spent 27 years in prison and became his country's first Black president."},
]

## 2. The models

Two **small** models from the [Hugging Face hub](https://huggingface.co/models?library=sentence-transformers). Same job, different upbringing:

| | parameters | trained on |
|---|---|---|
| [`all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) | 22M | **English only** |
| [`paraphrase-multilingual-MiniLM-L12-v2`](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2) | 118M | **50+ languages** |

Any other model name from the hub works here too — that's exercise 5.

In [ ]:
MODEL_A = "sentence-transformers/all-MiniLM-L6-v2"
MODEL_B = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

## 3. Two world views

Each model reads the same texts. Each dot below is one bio, placed on each model's own 2D map (hover a dot to read the bio).

In [ ]:
compare_models(ITEMS, MODEL_A, MODEL_B)

### What are you looking at?

- Each model turns a bio into a vector of 384 numbers; we squash those down to 2D. **The axes mean nothing — only distances do.** Close dots = "these texts look similar *to this model*".
- Do the three groups form clusters? In both models? Who ended up nearest to the data team, and why?
- 🔍 **Find the two Macrons.** Same bio, two languages. The multilingual model (right) sees them as *the same text* (similarity ≈ 0.95 — the dots overlap). The English-only model (left) only vaguely relates them (≈ 0.6), and mostly because of the shared names *Élysée* and *Paris* — exercise 3 pushes on exactly this weakness.
- Same texts, different training data → **a genuinely different view of the world**. Neither map is "the truth"; each is what the model learned to pay attention to.

*(Tip: `compare_models(ITEMS, MODEL_A, MODEL_B, method="tsne")` gives an alternative projection that exaggerates clusters.)*

## 4. Who is closest to whom?

The 2D map is a lossy summary. The matrix below is the real thing: the similarity the model assigns to **every pair** of texts (1.0 = identical view). Swap in `MODEL_A` and see which pairs change the most.

In [ ]:
similarity_heatmap(ITEMS, MODEL_B)

## 5. Your turn 🔬

Edit the cells above and re-run (the models stay cached, so re-runs are fast):

1. **Add yourself** to `ITEMS` (the `👇 ADD YOURSELF` slot). Which cluster do you land in? Who is your nearest neighbour — and do both models agree?
2. **Rewrite your bio as a cartoon character** ("…lives in a pineapple…") and watch yourself migrate across the map. What does that tell you about what the model reads: your name, or your words?
3. **Translate a bio that shares no names between languages** — e.g. add SpongeBob in French: *"Éponge de mer jaune et optimiste qui vit dans un ananas sous la mer et retourne des burgers."* For the English-only model the two SpongeBobs become near-strangers (similarity ≈ 0.3 — check the heatmap!), while the multilingual one keeps them together (≈ 0.7). Which model would you trust to search French customer feedback?
4. **Fool the map**: write one sentence that lands exactly *between* the politicians and the cartoons (a president of Springfield?).
5. **Swap a model**: put any name from [huggingface.co/models](https://huggingface.co/models?library=sentence-transformers) into `MODEL_A` — e.g. `"intfloat/multilingual-e5-small"` — and see how *its* world differs.

### What you just learned

An embedding is a model's world view: texts become points, meaning becomes distance. Different training data or objective → different geometry. This exact mechanism is what powers semantic search, recommendation, clustering and RAG — there, too, the model's world view (not yours) decides what counts as "similar".